# XGBoost: topology-informed hydropower unit commitment

Group number: TODO. Full names and student IDs: TODO. Kaggle team: TODO.

Fourteen generator-specific XGBoost classifiers predict all 168 hourly ON probabilities.
The notebook starts from supplied raw data and contains every runtime helper.
Modes: check, smoke, validate, holdout, submit. Check fits no models. Development
uses purged2020/2021 cases and Logloss stopping. Holdout uses frozen budgets on2022,
which has prior project exposure; submit refits all2922 cases without test stopping.

Controlled feature presets add directed hydraulic topology, same-case price shape,
gross-head/efficiency descriptors and timestamp-aligned past history. The allocation
preset adds a deterministic twelve-hour network-flow relaxation; its projected
storage is calculated from available inputs, never observed future storage. It is
an approximation without full tunnel pressure balance, changing head, flow penalties
or binary start/stop physics, rather than an exact reproduction of SHOP. No future
observed storage, UC labels, optimizer runtime or external data enter predictors.
Connectivity/pressure/head are proxies, not optimized releases or a hydraulic simulation.
Use gain importance with correlation/causality limitations. Hyperparameters and
performance are experimental until measured; record unsuccessful trials as well.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from time import perf_counter
import gc
import hashlib
import json
import platform
import re
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

ROOT = Path.cwd()
RUN_MODE = "check"
EXPERIMENT = 'topology_allocation'
MODEL_FAMILY = "xgboost"
MODEL_STRATEGY = "per_generator"
SEED = 42
THREADS = 4
HOURS = 168
DEADLINE_UTC = None
EXPERIMENTS = {
    "decision_reference": {"groups": [], "version": "hydro-decision-v1"},
    "topology": {"groups": ["topology"], "version": "xgb-topology-v1"},
    "topology_shape": {"groups": ["topology", "shape"], "version": "xgb-topology-shape-v1"},
    "topology_physics": {"groups": ["topology", "physics"], "version": "xgb-topology-physics-v1"},
    "topology_history": {"groups": ["topology", "history"], "version": "xgb-topology-history-v1"},
    "topology_combined": {"groups": ["topology", "shape", "physics", "history"], "version": "xgb-topology-combined-v1"},
    "topology_allocation": {"groups": ["topology", "allocation"], "version": "xgb-topology-allocation-v1"},
}
if EXPERIMENT not in EXPERIMENTS:
    raise ValueError("Unknown XGBoost feature experiment.")
FEATURE_VERSION = EXPERIMENTS[EXPERIMENT]["version"]
ADDITIONAL_GROUPS = set(EXPERIMENTS[EXPERIMENT]["groups"])
DECISION_GROUPS = {"economics", "water_balance"}
FOLDS = [("2020-01-01", "2021-01-01"), ("2021-01-01", "2022-01-01")]
HOLDOUT_START = pd.Timestamp("2022-01-01", tz="UTC")
EARLY_STOPPING = 80
MAX_ITERATIONS = 1200
BASE_PARAMETERS = {
    "objective": "binary:logistic", "eval_metric": "logloss",
    "tree_method": "hist", "device": "cpu", "max_bin": 128,
    "max_depth": 7, "learning_rate": 0.05, "min_child_weight": 5,
    "reg_lambda": 10.0, "reg_alpha": 0.0, "subsample": 0.8,
    "colsample_bytree": 0.8, "base_score": 0.5,
}
SELECTED_CONFIG = {'model_family': 'xgboost',
 'experiment': 'topology_allocation',
 'model_strategy': 'per_generator',
 'feature_version': 'xgb-topology-allocation-v1',
 'feature_signature': 'd08f69d2d991d7e227d3f0447ac1a9b16736704ccad26fb9fd74f608aa86f4da',
 'seed': 42,
 'parameters': {'objective': 'binary:logistic',
                'eval_metric': 'logloss',
                'tree_method': 'hist',
                'device': 'cpu',
                'max_bin': 128,
                'max_depth': 7,
                'learning_rate': 0.05,
                'min_child_weight': 5,
                'reg_lambda': 10.0,
                'reg_alpha': 0.0,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'base_score': 0.5},
 'iterations': {'Hogga_G1': 268,
                'Tokke_G1': 263,
                'Tokke_G2': 168,
                'Tokke_G3': 255,
                'Tokke_G4': 156,
                'Lio_G1': 243,
                'Byrte_G1': 293,
                'Vinje_G1': 262,
                'Vinje_G2': 191,
                'Vinje_G3': 399,
                'Haukeli_G1': 262,
                'Songa_G1': 191,
                'Kjela_G1': 515,
                'Vesle Kjela_G1': 318}}
if RUN_MODE not in {"check", "smoke", "validate", "holdout", "submit"}:
    raise ValueError("Unknown RUN_MODE.")
if not (ROOT / "prediction_mapping.csv").is_file():
    raise FileNotFoundError("Run from the project root with the provided raw data.")
if not 1 <= THREADS <= 4:
    raise ValueError("Use one to four CPU threads.")
NOTEBOOK_STARTED = perf_counter()
VERSIONS = {"python": platform.python_version()}
for package in ("numpy", "pandas", "PyYAML", "xgboost", "scipy", "scikit-learn", "matplotlib"):
    try:
        VERSIONS[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        VERSIONS[package] = "not installed"
print("Mode:", RUN_MODE, "Model:", MODEL_FAMILY, "Experiment:", EXPERIMENT)
print("Package versions:", VERSIONS)

In [ ]:
RAW_FILES = {
    "labels": ROOT / "data/kernel/Unit_commitment_decisions.csv",
    "mapping": ROOT / "prediction_mapping.csv",
    "template": ROOT / "sample_submission.csv",
    "prices": ROOT / "data/kernel/Historical_day_ahead_price_2015_2025.csv",
    "inflows": ROOT / "data/kernel/Historical_inflow_1958_2025.csv",
    "volumes": ROOT / "data/kernel/Historical_volume_2015_2024.csv",
    "water_values": ROOT / "data/kernel/Synthetic_water_value_2015_2024.csv",
    "min_flow": ROOT / "data/extended/Constraint_min_flow.csv",
    "min_volume": ROOT / "data/extended/Constraint_min_volume.csv",
    "topology": ROOT / "data/extended/Tokke_Vinje_topology.yaml",
}
for path in RAW_FILES.values():
    if not path.is_file():
        raise FileNotFoundError(path)
train_cases = pd.read_csv(RAW_FILES["labels"])
test_cases = pd.read_csv(RAW_FILES["mapping"])
template = pd.read_csv(RAW_FILES["template"])
for cases in (train_cases, test_cases):
    cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True, errors="raise")
    if cases.empty or cases["Run No"].isna().any() or not cases["Run No"].is_unique:
        raise ValueError("Cases require nonempty, unique, nonmissing Run No values.")
    if cases["starttime"].isna().any() or not cases["starttime"].is_unique:
        raise ValueError("Case start times must be unique and nonmissing.")
    if not cases["starttime"].equals(cases["starttime"].dt.normalize()):
        raise ValueError("Expected midnight UTC case starts.")
if not template["Run No"].is_unique or set(template["Run No"]) != set(test_cases["Run No"]):
    raise ValueError("Submission IDs must match the test mapping exactly.")
if set(train_cases["Run No"]) & set(test_cases["Run No"]):
    raise ValueError("Training and test case IDs overlap.")
train_cases = train_cases.sort_values("starttime").reset_index(drop=True)
test_cases = test_cases.set_index("Run No").loc[template["Run No"]].reset_index()
LABELS = template.columns.drop("Run No").tolist()
TARGETS = {}
for position, name in enumerate(LABELS):
    match = re.fullmatch(r"result_committed_(.+_G\d+)_t(\d+)", name)
    if match is None:
        raise ValueError(f"Unexpected target column: {name}")
    generator, hour = match.group(1), int(match.group(2))
    if hour in TARGETS.setdefault(generator, {}):
        raise ValueError(f"Duplicate generator/hour: {generator}, {hour}")
    TARGETS[generator][hour] = (name, position)
GENERATORS = list(TARGETS)
if len(GENERATORS) != 14 or len(LABELS) != 14 * HOURS:
    raise ValueError("Expected 14 generators and 2352 targets.")
for generator in GENERATORS:
    if set(TARGETS[generator]) != set(range(HOURS)):
        raise ValueError(f"Incomplete horizon for {generator}")
TARGET_COLUMNS = {g: [TARGETS[g][h][0] for h in range(HOURS)] for g in GENERATORS}
TARGET_POSITIONS = {g: [TARGETS[g][h][1] for h in range(HOURS)] for g in GENERATORS}
if not set(LABELS).issubset(train_cases) or not train_cases[LABELS].isin([0, 1]).all().all():
    raise ValueError("All target columns must be present with binary, nonmissing labels.")

def labels_for(cases, generator):
    return cases[TARGET_COLUMNS[generator]].to_numpy(dtype=np.uint8).reshape(-1)

all_starts = pd.DatetimeIndex(pd.concat([train_cases["starttime"], test_cases["starttime"]]))
FIRST_START = all_starts.min()
LAST_END = all_starts.max() + pd.Timedelta(hours=HOURS)
print(f"{len(train_cases)} training cases; {len(test_cases)} test cases; {len(LABELS)} targets")
display(pd.DataFrame({"generator": GENERATORS,
    "training_on_fraction": [float(labels_for(train_cases, g).mean()) for g in GENERATORS]}))

In [ ]:
def read_timeseries(key):
    frame = pd.read_csv(RAW_FILES[key])
    dates = pd.to_datetime(frame.pop("date"), utc=True, errors="raise")
    if dates.isna().any() or dates.duplicated().any():
        raise ValueError(f"Invalid or duplicate timestamps in {key}")
    frame.index = pd.DatetimeIndex(dates, name="date")
    frame = frame.sort_index().loc[FIRST_START:LAST_END]
    frame = frame.apply(pd.to_numeric, errors="raise").astype(np.float32)
    if frame.empty or not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f"Empty or nonfinite input: {key}")
    return frame

tables = {key: read_timeseries(key) for key in
          ("prices", "inflows", "volumes", "water_values", "min_flow", "min_volume")}
if list(tables["prices"].columns) != ["Day-ahead price (EUR/MWh)"]:
    raise ValueError("Unexpected electricity price column.")
with RAW_FILES["topology"].open(encoding="utf-8") as stream:
    topology = yaml.safe_load(stream)
RESERVOIRS = list(tables["volumes"].columns)
if len(RESERVOIRS) != 17:
    raise ValueError("Expected 17 reservoirs.")
for key in ("inflows", "water_values"):
    if not set(RESERVOIRS).issubset(tables[key].columns):
        raise ValueError(f"Missing reservoir columns in {key}")
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}
if any(v <= 0 for v in CAPACITY.values()):
    raise ValueError("Reservoir capacities must be positive.")
for key in ("inflows", "volumes", "min_flow", "min_volume"):
    if (tables[key].to_numpy() < 0).any():
        raise ValueError(f"Unexpected negative water quantity in {key}")

hourly_required = pd.date_range(FIRST_START, LAST_END, freq="h", inclusive="left")
for key in ("prices", "inflows", "min_flow", "min_volume"):
    used = tables[key].loc[tables[key].index < LAST_END]
    if not used.index.equals(hourly_required):
        raise ValueError(f"{key} must contain exactly one record per required hour.")
for key, timestamps in (("volumes", all_starts),
                        ("water_values", all_starts + pd.Timedelta(hours=HOURS))):
    if not timestamps.difference(tables[key].index).empty:
        raise ValueError(f"Missing required daily records in {key}")

connections = topology["connections"]
def supplying_reservoirs(name, kind, visited=None):
    visited = set() if visited is None else visited
    node = (kind, name)
    if node in visited:
        raise ValueError("Cycle in intake topology.")
    if kind == "reservoir":
        return {name}
    result = set()
    for edge in connections:
        if edge["to"] == name and edge["to_type"] == kind and edge["from_type"] in {"reservoir", "tunnel"}:
            result |= supplying_reservoirs(edge["from"], edge["from_type"], visited | {node})
    return result

FEEDERS = {}
for generator in GENERATORS:
    plants = [e["to"] for e in connections if e["from"] == generator
              and e["from_type"] == "generator" and e["to_type"] == "plant"]
    if len(plants) != 1:
        raise ValueError(f"Expected one plant for {generator}")
    FEEDERS[generator] = sorted(supplying_reservoirs(plants[0], "plant"))
    if not FEEDERS[generator] or not set(FEEDERS[generator]).issubset(RESERVOIRS):
        raise ValueError(f"Unknown supplying reservoirs for {generator}")
display(pd.DataFrame({"generator": GENERATORS,
                      "intake_reservoirs": [", ".join(FEEDERS[g]) for g in GENERATORS]}))
display(pd.DataFrame([{"input": k, "rows_in_task_range": len(v), "columns": len(v.columns),
                       "start": str(v.index.min()), "end": str(v.index.max())}
                      for k, v in tables.items()]))

# Traverse every supplied hydraulic predecessor, including river reaches and plants.
# A visited set prevents repeated counting at merges or cycles. These are possible
# upstream connections, not evidence that all upstream water arrives in this horizon.
def upstream_reservoirs(generator):
    plant = next(e["to"] for e in connections if e["from"] == generator
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    pending, visited, reservoirs = [("plant", plant)], set(), set()
    while pending:
        kind, name = pending.pop()
        if (kind, name) in visited:
            continue
        visited.add((kind, name))
        if kind == "reservoir":
            reservoirs.add(name)
        pending.extend((edge["from_type"], edge["from"]) for edge in connections
                       if edge["to_type"] == kind and edge["to"] == name
                       and edge["from_type"] in {"reservoir", "river", "tunnel", "plant"})
    return sorted(reservoirs)

UPSTREAM = {g: upstream_reservoirs(g) for g in GENERATORS}
OUTLET_ELEVATION = {}
for g in GENERATORS:
    plant = next(e["to"] for e in connections if e["from"] == g
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    OUTLET_ELEVATION[g] = float(topology["model"]["plant"][plant]["outlet_line"])
    if not set(FEEDERS[g]).issubset(UPSTREAM[g]) or not set(UPSTREAM[g]).issubset(RESERVOIRS):
        raise ValueError(f"Invalid upstream topology for {g}")
if FEATURE_VERSION == "hydro-context-v2":
    display(pd.DataFrame({"generator": GENERATORS,
                          "all_upstream_reservoirs": [", ".join(UPSTREAM[g]) for g in GENERATORS]}))

# Directed hydraulic edges exclude generator membership edges. Stop at the first
# receiving reservoir; never interpret graph connectivity as realized routing.
HYDRAULIC_KINDS = {"reservoir", "plant", "river", "tunnel"}
FORWARD, BACKWARD = {}, {}
for edge in connections:
    if edge["from_type"] in HYDRAULIC_KINDS and edge["to_type"] in HYDRAULIC_KINDS:
        left = (edge["from_type"], edge["from"])
        right = (edge["to_type"], edge["to"])
        FORWARD.setdefault(left, set()).add(right)
        BACKWARD.setdefault(right, set()).add(left)
PLANT_FOR = {g: next(e["to"] for e in connections if e["from"] == g
                     and e["from_type"] == "generator" and e["to_type"] == "plant")
             for g in GENERATORS}
PLANT_GENERATORS = {p: [g for g in GENERATORS if PLANT_FOR[g] == p]
                    for p in topology["model"]["plant"]}
PLANT_FEEDERS = {p: FEEDERS[gens[0]] for p, gens in PLANT_GENERATORS.items()}

def reachable(start, adjacency, stop_at_reservoir=False):
    pending, seen = list(adjacency.get(start, [])), set()
    while pending:
        node = pending.pop()
        if node == start or node in seen:
            continue
        seen.add(node)
        if not (stop_at_reservoir and node[0] == "reservoir"):
            pending.extend(adjacency.get(node, []))
    return seen

RECEIVERS = {p: sorted(name for kind, name in reachable(("plant", p), FORWARD, True)
                       if kind == "reservoir") for p in PLANT_GENERATORS}
UPSTREAM_PLANTS = {p: sorted(name for kind, name in reachable(("plant", p), BACKWARD)
                             if kind == "plant") for p in PLANT_GENERATORS}
DOWNSTREAM_PLANTS = {p: sorted(name for kind, name in reachable(("plant", p), FORWARD)
                               if kind == "plant") for p in PLANT_GENERATORS}
LEVEL_CURVES = {r: (np.asarray(topology["model"]["reservoir"][r]["vol_head"]["x"], dtype=float),
                    np.asarray(topology["model"]["reservoir"][r]["vol_head"]["y"], dtype=float))
                for r in RESERVOIRS}
for r, (x, y) in LEVEL_CURVES.items():
    if len(x) != len(y) or not np.all(np.diff(x) > 0) or not np.all(np.diff(y) >= 0):
        raise ValueError(f"Invalid volume-level curve: {r}")
GEN_SPECS = topology["model"]["generator"]
display(pd.DataFrame([{"plant": p, "intakes": PLANT_FEEDERS[p], "receivers": RECEIVERS[p],
                       "upstream_plants": UPSTREAM_PLANTS[p], "downstream_plants": DOWNSTREAM_PLANTS[p]}
                      for p in PLANT_GENERATORS]))

In [ ]:
def economic_context(margin, price):
    """Whole-horizon price/value descriptors, not a dispatch profitability solver."""
    margin, price = np.asarray(margin, dtype=np.float64), np.asarray(price, dtype=np.float64)
    n, width = margin.shape
    if width != HOURS or price.shape != margin.shape:
        raise ValueError("Economic context requires one complete horizon per row.")
    positive = margin > 0
    positive_margin = np.maximum(margin, 0)
    left_length = np.zeros_like(margin)
    right_length = np.zeros_like(margin)
    left_sum = np.zeros_like(margin)
    right_sum = np.zeros_like(margin)
    next_positive = np.full_like(margin, HOURS)
    previous_positive = np.full_like(margin, HOURS)
    # Each recurrence resets independently for every scheduling case.
    for h in range(width):
        left_length[:, h] = np.where(positive[:, h], (left_length[:, h-1] if h else 0) + 1, 0)
        left_sum[:, h] = np.where(positive[:, h], (left_sum[:, h-1] if h else 0) + margin[:, h], 0)
        previous_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((previous_positive[:, h-1] if h else HOURS) + 1, HOURS))
    for h in range(width - 1, -1, -1):
        right_length[:, h] = np.where(positive[:, h], (right_length[:, h+1] if h < width-1 else 0) + 1, 0)
        right_sum[:, h] = np.where(positive[:, h], (right_sum[:, h+1] if h < width-1 else 0) + margin[:, h], 0)
        next_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((next_positive[:, h+1] if h < width-1 else HOURS) + 1, HOURS))
    suffix_count = np.arange(width, 0, -1)
    suffix_positive = positive[:, ::-1].cumsum(1)[:, ::-1]
    suffix_margin = positive_margin[:, ::-1].cumsum(1)[:, ::-1]
    result = {
        "economic_margin_over_week_price_scale": margin / (price.std(1, keepdims=True) + 1.0),
        "economic_positive_streak_elapsed_h": left_length,
        "economic_positive_streak_remaining_h": right_length,
        "economic_positive_streak_total_h": left_length + right_length - positive,
        "economic_positive_streak_margin_sum": left_sum + right_sum - positive_margin,
        "economic_hours_to_next_positive": next_positive,
        "economic_hours_since_previous_positive": previous_positive,
        "economic_positive_fraction_remaining": suffix_positive / suffix_count,
        "economic_positive_margin_remaining_mean": suffix_margin / suffix_count,
        "economic_better_remaining_margin": np.maximum.accumulate(margin[:, ::-1], axis=1)[:, ::-1] - margin,
    }
    indices = np.arange(width)
    for window in (6, 24):
        end = np.minimum(indices + window, width)
        start = np.maximum(indices - window + 1, 0)
        prefix = np.concatenate([np.zeros((n, 1)), margin.cumsum(1)], axis=1)
        result[f"economic_next_{window}h_margin_mean"] = (prefix[:, end] - prefix[:, indices]) / (end - indices)
        result[f"economic_previous_{window}h_margin_mean"] = (prefix[:, indices + 1] - prefix[:, start]) / (indices + 1 - start)
    return result


def water_context(initial, cumulative, remaining, limits, capacities):
    """Local natural-inflow scenarios with zero releases/transfers, not future storage."""
    initial = np.asarray(initial, dtype=np.float64)
    cumulative = np.asarray(cumulative, dtype=np.float64)
    remaining = np.asarray(remaining, dtype=np.float64)
    limits = np.asarray(limits, dtype=np.float64)
    capacities = np.asarray(capacities, dtype=np.float64)
    if cumulative.shape != remaining.shape or limits.shape != cumulative.shape or             cumulative.shape[1:] != (HOURS, len(capacities)) or initial.shape != (len(cumulative), len(capacities)):
        raise ValueError("Invalid water-context dimensions.")
    if (capacities <= 0).any():
        raise ValueError("Reservoir capacities must be positive.")
    capacity = capacities.sum()
    start = initial[:, None, :]
    no_release = start + cumulative
    end_no_release = no_release + remaining
    headroom = np.maximum(capacities - start, 0)
    # A 1% capacity floor bounds the ratio near a full reservoir; apply per reservoir
    # before aggregation so spare space elsewhere does not cancel local pressure.
    pressure = np.log1p((cumulative + remaining) / np.maximum(headroom, 0.01 * capacities))
    future_floor = np.maximum.accumulate(limits[:, ::-1, :], axis=1)[:, ::-1, :]
    return {
        "water_week_inflow_to_headroom_log_max": pressure.max(2),
        "water_week_inflow_to_headroom_log_weighted": (pressure * capacities).sum(2) / capacity,
        "water_no_release_excess_fraction": np.maximum(no_release - capacities, 0).sum(2) / capacity,
        "water_end_no_release_excess_fraction": np.maximum(end_no_release - capacities, 0).sum(2) / capacity,
        "water_no_release_excess_max_fraction": np.maximum((no_release - capacities) / capacities, 0).max(2),
        "water_initial_margin_to_floor_fraction": (start - limits).sum(2) / capacity,
        "water_initial_margin_to_future_floor_fraction": (start - future_floor).sum(2) / capacity,
        "water_no_release_floor_deficit_fraction": np.maximum(limits - no_release, 0).sum(2) / capacity,
        "water_no_release_min_margin_fraction": ((no_release - limits) / capacities).min(2),
    }


def lookup(key, timestamps, columns=None):
    frame = tables[key] if columns is None else tables[key][columns]
    selected = frame.reindex(pd.DatetimeIndex(timestamps))
    if selected.isna().any().any():
        raise ValueError(f"Missing values while looking up {key}")
    return selected.to_numpy(dtype=np.float32)

def feature_matrix(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    if len(starts) == 0:
        raise ValueError("Cannot construct features for zero cases.")
    n = len(starts)
    # Use datetime arithmetic rather than assuming pandas timestamps use nanoseconds.
    timestamps = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h")
    price = lookup("prices", timestamps).reshape(n, HOURS)
    initial = lookup("volumes", starts, RESERVOIRS)
    terminal = lookup("water_values", starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflows = lookup("inflows", timestamps).reshape(n, HOURS, -1)
    features = {}

    def add(name, values):
        values = np.asarray(values, dtype=np.float32)
        if values.shape == (n,):
            values = np.repeat(values, HOURS)
        elif values.shape == (n, HOURS):
            values = values.reshape(-1)
        if values.shape != (n * HOURS,) or not np.isfinite(values).all():
            raise ValueError(f"Invalid feature {name}: {values.shape}")
        features[name] = values

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", timestamps.hour.to_numpy())
    add("day_of_week", timestamps.dayofweek.to_numpy())
    add("month", timestamps.month.to_numpy())
    add("day_of_year_sin", np.sin(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("day_of_year_cos", np.cos(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("price", price)
    for stat, values in (("mean", price.mean(1)), ("std", price.std(1)),
                         ("min", price.min(1)), ("max", price.max(1))):
        add(f"price_week_{stat}", values)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, values in (("mean", daily.mean(2)), ("min", daily.min(2)), ("max", daily.max(2))):
        add(f"price_day_{stat}", np.repeat(values, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add("price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    # Clip at the case boundary, not at the global dataset boundary.
    for offset in (-24, -6, -1, 1, 6, 24):
        add(f"price_offset_{offset:+d}", price[:, np.clip(np.arange(HOURS) + offset, 0, HOURS - 1)])
    for j, reservoir in enumerate(RESERVOIRS):
        add(f"initial_volume__{reservoir}", initial[:, j])
        add(f"initial_fraction__{reservoir}", initial[:, j] / CAPACITY[reservoir])
        add(f"terminal_value__{reservoir}", terminal[:, j])
        add(f"price_minus_value__{reservoir}", price - terminal[:, j, None])
    for j, name in enumerate(tables["inflows"].columns):
        flow = inflows[:, :, j]
        cumulative = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cumulative_Mm3__{name}", cumulative)
        add(f"inflow_remaining_Mm3__{name}", cumulative[:, -1:] - cumulative)
    for key in ("min_flow", "min_volume"):
        limits = lookup(key, timestamps).reshape(n, HOURS, -1)
        for j, name in enumerate(tables[key].columns):
            add(f"{key}__{name}", limits[:, :, j])
            add(f"{key}_week_max__{name}", limits[:, :, j].max(1))
            if key == "min_volume":
                add(f"initial_margin_to_min__{name}", initial[:, RESERVOIRS.index(name), None] - limits[:, :, j])
    if FEATURE_VERSION == "hydro-context-v2":
        # Rolling summaries stay strictly inside each supplied planning horizon.
        add("price_past_24h_mean", pd.DataFrame(price.T).rolling(24, min_periods=1).mean().to_numpy().T)
        add("price_next_24h_mean", pd.DataFrame(price[:, ::-1].T).rolling(24, min_periods=1).mean().to_numpy().T[:, ::-1])
        add("price_suffix_max", np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        add("price_suffix_min", np.minimum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        for j, reservoir in enumerate(RESERVOIRS):
            curve = topology["model"]["reservoir"][reservoir]["vol_head"]
            curve_x, curve_y = np.asarray(curve["x"]), np.asarray(curve["y"])
            if len(curve_x) != len(curve_y) or not np.all(np.diff(curve_x) > 0):
                raise ValueError(f"Invalid volume-level curve for {reservoir}")
            # np.interp clips outside the supplied curve: a static descriptor only.
            add(f"initial_level_m__{reservoir}", np.interp(initial[:, j], curve_x, curve_y))
    return pd.DataFrame(features, dtype=np.float32)

def generator_features(base, generator):
    # Copy only one generator's matrix at a time; never materialize all 14 together.
    X = base.copy()
    feeders = FEEDERS[generator]
    capacity = sum(CAPACITY[r] for r in feeders)
    total_initial = sum(base[f"initial_volume__{r}"] for r in feeders)
    value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in feeders) / capacity
    cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in feeders)
    remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in feeders)
    X["local_initial_fraction"] = total_initial / capacity
    X["local_terminal_value_capacity_weighted"] = value
    X["local_price_minus_terminal_value"] = base["price"] - value
    X["local_inflow"] = sum(base[f"inflow__{r}"] for r in feeders)
    X["local_cumulative_inflow_fraction"] = cumulative / capacity
    X["local_remaining_inflow_fraction"] = remaining / capacity
    X["local_available_water_proxy_fraction"] = (total_initial + cumulative + remaining) / capacity
    if FEATURE_VERSION == "hydro-context-v2":
        upstream = UPSTREAM[generator]
        upstream_capacity = sum(CAPACITY[r] for r in upstream)
        upstream_initial = sum(base[f"initial_volume__{r}"] for r in upstream)
        upstream_cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in upstream)
        upstream_remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in upstream)
        upstream_value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in upstream) / upstream_capacity
        X["upstream_initial_fraction"] = upstream_initial / upstream_capacity
        X["upstream_terminal_value_capacity_weighted"] = upstream_value
        X["upstream_price_minus_value"] = base["price"] - upstream_value
        X["upstream_inflow"] = sum(base[f"inflow__{r}"] for r in upstream)
        X["upstream_cumulative_inflow_fraction"] = upstream_cumulative / upstream_capacity
        X["upstream_remaining_inflow_fraction"] = upstream_remaining / upstream_capacity
        X["upstream_available_water_proxy_fraction"] = (upstream_initial + upstream_cumulative + upstream_remaining) / upstream_capacity
        initial_level = sum(base[f"initial_level_m__{r}"] * CAPACITY[r] for r in feeders) / capacity
        X["local_initial_gross_head_proxy_m"] = initial_level - OUTLET_ELEVATION[generator]
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        positive = (margin > 0).astype(np.float32)
        X["local_hours_above_value_remaining"] = positive[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
        X["local_positive_margin_remaining"] = np.maximum(margin, 0)[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
    extra = {}
    if "economics" in DECISION_GROUPS:
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        extra.update(economic_context(margin, base["price"].to_numpy().reshape(-1, HOURS)))
    if "water_balance" in DECISION_GROUPS:
        def cube(prefix):
            return np.stack([base[f"{prefix}__{r}"].to_numpy().reshape(-1, HOURS) for r in feeders], axis=2)
        initial = cube("initial_volume")[:, 0, :]
        cumulative_cube = cube("inflow_cumulative_Mm3")
        remaining_cube = cube("inflow_remaining_Mm3")
        floor = np.stack([base[f"min_volume__{r}"].to_numpy().reshape(-1, HOURS)
                          if f"min_volume__{r}" in base else np.zeros_like(cumulative_cube[:, :, 0])
                          for r in feeders], axis=2)
        extra.update(water_context(initial, cumulative_cube, remaining_cube, floor,
                                   [CAPACITY[r] for r in feeders]))
    if extra:
        X = pd.concat([X, pd.DataFrame({name: values.reshape(-1).astype(np.float32)
                                       for name, values in extra.items()}, index=X.index)], axis=1)
    if not np.isfinite(X.to_numpy()).all():
        raise ValueError("Features must be finite.")
    return X.astype(np.float32, copy=False)
reference_feature_matrix = feature_matrix
reference_generator_features = generator_features

def feature_matrix(cases):
    frame = reference_feature_matrix(cases)
    # Internal alignment metadata never enters the numeric feature matrix.
    frame.attrs["case_starts"] = pd.DatetimeIndex(cases["starttime"])
    return frame

def state(base, reservoirs, prefix):
    return np.stack([base[f"{prefix}__{r}"].to_numpy().reshape(-1, HOURS)
                     for r in reservoirs], axis=2)

def plant_state(base, plant):
    feeders = PLANT_FEEDERS[plant]
    weights = np.array([CAPACITY[r] for r in feeders], dtype=np.float64)
    volumes = state(base, feeders, "initial_volume")
    values = state(base, feeders, "terminal_value")
    level = np.stack([np.interp(volumes[:, :, i], *LEVEL_CURVES[r]) for i, r in enumerate(feeders)], axis=2)
    intake_level = np.average(level, weights=weights, axis=2)
    receiver = RECEIVERS[plant]
    outlet = float(topology["model"]["plant"][plant]["outlet_line"])
    tailwater = np.full_like(intake_level, outlet)
    if receiver:
        receiver_levels = np.stack([np.interp(base[f"initial_volume__{r}"].to_numpy().reshape(-1, HOURS),
                                              *LEVEL_CURVES[r]) for r in receiver], axis=2)
        tailwater = np.maximum(tailwater, receiver_levels.max(axis=2))
    return {"head": np.maximum(intake_level-tailwater, 1.0),
            "intake_level": intake_level, "tailwater": tailwater,
            "fraction": volumes.sum(axis=2)/weights.sum(),
            "value": np.average(values, weights=weights, axis=2)}

def topology_features(base, generator):
    plant = PLANT_FOR[generator]
    feeders, upstream, receiver = FEEDERS[generator], UPSTREAM[generator], RECEIVERS[plant]
    local = plant_state(base, plant)
    price = base["price"].to_numpy().reshape(-1, HOURS)
    capacity = np.array([CAPACITY[r] for r in upstream])
    volumes = state(base, upstream, "initial_volume")
    fractions = volumes / capacity
    terminal = state(base, upstream, "terminal_value")
    cumulative = state(base, upstream, "inflow_cumulative_Mm3")
    remaining = state(base, upstream, "inflow_remaining_Mm3")
    root_capacity = capacity.sum()
    pressure = (cumulative+remaining)/np.maximum(capacity-volumes, 0.01*capacity)
    extra = {
        "topo_initial_intake_level_m": local["intake_level"],
        "topo_initial_tailwater_m": local["tailwater"],
        "topo_initial_gross_head_m": local["head"],
        "topo_head_relative_reference": local["head"]/np.mean([c["ref"] for c in GEN_SPECS[generator]["turb_eff_curves"]]),
        "topo_upstream_initial_fraction": volumes.sum(2)/root_capacity,
        "topo_upstream_min_fraction": fractions.min(2),
        "topo_upstream_max_fraction": fractions.max(2),
        "topo_upstream_std_fraction": fractions.std(2),
        "topo_upstream_value": np.average(terminal, weights=capacity, axis=2),
        "topo_upstream_value_min": terminal.min(2),
        "topo_upstream_value_max": terminal.max(2),
        "topo_upstream_value_std": terminal.std(2),
        "topo_upstream_inflow": state(base, upstream, "inflow").sum(2),
        "topo_upstream_cumulative_fraction": cumulative.sum(2)/root_capacity,
        "topo_upstream_remaining_fraction": remaining.sum(2)/root_capacity,
        "topo_upstream_water_pressure_max": np.log1p(pressure).max(2),
        "topo_upstream_excess_fraction": np.maximum(volumes+cumulative+remaining-capacity, 0).sum(2)/root_capacity,
        "topo_upstream_price_minus_value": price-np.average(terminal, weights=capacity, axis=2),
    }
    for label, reservoirs in [("receiver", receiver), ("remote_upstream", sorted(set(upstream)-set(feeders)))]:
        if reservoirs:
            cap = np.array([CAPACITY[r] for r in reservoirs])
            vol = state(base, reservoirs, "initial_volume")
            wv = np.average(state(base, reservoirs, "terminal_value"), weights=cap, axis=2)
            frac = vol.sum(2)/cap.sum()
            inflow = state(base, reservoirs, "inflow").sum(2)
            free = np.maximum(cap-vol,0).sum(2)/cap.sum()
        else:
            frac = wv = inflow = free = np.zeros_like(price)
        extra.update({f"topo_{label}_fraction": frac, f"topo_{label}_value": wv,
                      f"topo_{label}_inflow": inflow, f"topo_{label}_free_fraction": free,
                      f"topo_local_minus_{label}_value": local["value"]-wv})
    for direction, related in [("upstream", UPSTREAM_PLANTS[plant]), ("downstream", DOWNSTREAM_PLANTS[plant])]:
        related_states = [plant_state(base,p) for p in related]
        margins = np.stack([price-s["value"] for s in related_states],axis=2) if related_states else np.zeros((*price.shape,1))
        fills = np.stack([s["fraction"] for s in related_states],axis=2) if related_states else np.zeros((*price.shape,1))
        extra.update({f"topo_{direction}_plant_margin_mean": margins.mean(2),
                      f"topo_{direction}_plant_margin_min": margins.min(2),
                      f"topo_{direction}_plant_margin_max": margins.max(2),
                      f"topo_{direction}_plant_positive_fraction": (margins>0).mean(2),
                      f"topo_{direction}_plant_fill_mean": fills.mean(2),
                      f"topo_{direction}_plant_fill_min": fills.min(2),
                      f"topo_{direction}_plant_fill_max": fills.max(2)})
    for r in RESERVOIRS:
        extra[f"topo_initial_level_m__{r}"] = np.interp(base[f"initial_volume__{r}"].to_numpy().reshape(-1,HOURS),*LEVEL_CURVES[r])
    # The source of each min-flow constraint is explicit in the hydraulic graph.
    for label, reservoirs in [("intake",set(feeders)),("upstream",set(upstream))]:
        river_names = {name for r in reservoirs for kind,name in FORWARD.get(("reservoir",r),[]) if kind=="river"}
        requirement = sum((base[f"min_flow__{r}"].to_numpy().reshape(-1,HOURS)
                           for r in river_names if f"min_flow__{r}" in base),np.zeros_like(price))
        flow = state(base,sorted(reservoirs),"inflow").sum(2)
        extra[f"topo_{label}_minflow_requirement"] = requirement
        extra[f"topo_{label}_natural_inflow_minus_requirement"] = flow-requirement
    return extra

def shape_features(base):
    price=base["price"].to_numpy().reshape(-1,HOURS)
    suffix=np.maximum.accumulate(price[:,::-1],axis=1)[:,::-1]
    prefix=np.maximum.accumulate(price,axis=1)
    rank=np.empty_like(price)
    for hour in range(HOURS):
        rank[:,hour]=(price[:,hour:]<price[:,hour:hour+1]).sum(1)/max(HOURS-hour-1,1)
    return {"shape_suffix_rank":rank, "shape_price_minus_suffix_max":price-suffix,
            "shape_price_over_suffix_max":price/(np.abs(suffix)+1),
            "shape_price_minus_prefix_max":price-prefix}

def efficiency(spec, head, power):
    head=np.asarray(head,dtype=np.float64)
    curves=sorted(spec["turb_eff_curves"],key=lambda c:c["ref"])
    refs=np.array([c["ref"] for c in curves])
    q=power*1e6/(1000*9.81*head*0.8)
    generator_curve=spec["gen_eff_curve"]
    eta_gen=np.interp(power,generator_curve["x"],np.array(generator_curve["y"])/100)
    for _ in range(4):
        etas=np.stack([np.interp(q,c["x"],np.array(c["y"])/100) for c in curves],axis=-1)
        upper=np.clip(np.searchsorted(refs,head,side="right"),1,max(1,len(refs)-1))
        if len(refs)==1:
            eta_turb=etas[...,0]
        else:
            lower=upper-1
            fraction=np.clip((head-refs[lower])/(refs[upper]-refs[lower]),0,1)
            eta_turb=np.take_along_axis(etas,lower[...,None],axis=-1)[...,0]*(1-fraction)+np.take_along_axis(etas,upper[...,None],axis=-1)[...,0]*fraction
        eta=np.clip(eta_gen*eta_turb,0.3,1.0)
        q=power*1e6/(1000*9.81*head*eta)
    return q,eta

def physics_features(base,generator):
    plant=PLANT_FOR[generator]
    local=plant_state(base,plant)
    head=local["head"][:,0]
    spec=GEN_SPECS[generator]
    reference=float(np.mean([c["ref"] for c in spec["turb_eff_curves"]]))
    price=base["price"].to_numpy().reshape(-1,HOURS)
    value=local["value"][:,0]
    result={}
    factors=[]
    for label,power in [("minimum",float(spec["p_min"])),("nominal",float(spec["p_nom"])),("maximum",float(spec["p_max"]))]:
        q,eta=efficiency(spec,head,power)
        q_ref,eta_ref=efficiency(spec,np.full_like(head,reference),power)
        factor=1000*9.81*head*eta/3600
        factor_ref=1000*9.81*reference*eta_ref/3600
        adjusted=price*(factor/factor_ref)[:,None]-value[:,None]
        result[f"physics_{label}_discharge_m3s"]=np.repeat(q[:,None],HOURS,axis=1)
        result[f"physics_{label}_energy_MWh_per_Mm3"]=np.repeat(factor[:,None],HOURS,axis=1)
        result[f"physics_{label}_adjusted_margin"]=adjusted
        if label=="minimum":
            positive=adjusted>0
            streak=np.zeros_like(adjusted)
            for t in range(HOURS):streak[:,t]=np.where(positive[:,t],(streak[:,t-1] if t else 0)+1,0)
            result["physics_positive_streak"]=streak
            result["physics_positive_opportunities_remaining"]=positive[:,::-1].cumsum(1)[:,::-1]
            result["physics_positive_margin_remaining"]=np.maximum(adjusted,0)[:,::-1].cumsum(1)[:,::-1]
            volume=sum(base[f"initial_volume__{r}"].to_numpy().reshape(-1,HOURS)[:,0] for r in FEEDERS[generator])
            weekflow=sum(base[f"inflow_week_mean__{r}"].to_numpy().reshape(-1,HOURS)[:,0] for r in FEEDERS[generator])
            result["physics_initial_minimum_run_hours_proxy"]=np.repeat((volume*1e6/(q*3600))[:,None],HOURS,axis=1)
            result["physics_natural_flow_fraction_minimum"]=np.repeat((weekflow/q)[:,None],HOURS,axis=1)
        factors.append(adjusted)
    result["physics_profitable_operating_points_fraction"]=np.mean(np.stack(factors,axis=2)>0,axis=2)
    return result

def history_features(base,generator):
    starts=base.attrs["case_starts"]
    feeders=FEEDERS[generator]
    cap=np.array([CAPACITY[r] for r in feeders])
    def past(key,days):
        requested=starts-pd.Timedelta(days=days)
        current=tables[key].reindex(starts)[feeders].to_numpy(dtype=float)
        previous=tables[key].reindex(requested)[feeders].to_numpy(dtype=float,copy=True)
        missing=~np.isfinite(previous).all(1)
        # Missing pre-dataset history falls back to that same case's current input.
        previous[missing]=current[missing]
        if key=="volumes":result=(current-previous).sum(1)/cap.sum()
        else:result=np.average(current-previous,weights=cap,axis=1)
        return result,missing.astype(float)
    result={}
    for days in [1,7,14]:
        for key in ["volumes","water_values"]:
            change,missing=past(key,days)
            result[f"history_{key}_{days}d_change"]=np.repeat(change[:,None],HOURS,axis=1)
            result[f"history_{key}_{days}d_missing"]=np.repeat(missing[:,None],HOURS,axis=1)
    current=tables["water_values"].reindex(starts)[feeders].to_numpy(dtype=float)
    terminal=np.average(state(base,feeders,"terminal_value"),weights=cap,axis=2)
    result["history_water_value_horizon_change"]=terminal-np.average(current,weights=cap,axis=1)[:,None]
    # Per-reservoir history preserves graph-node state changes that a single
    # intake-weighted average can hide, especially for assumed-constant intakes.
    current_volume=tables["volumes"].reindex(starts)[RESERVOIRS].to_numpy(dtype=float)
    for days in [1,7]:
        previous=tables["volumes"].reindex(starts-pd.Timedelta(days=days))[RESERVOIRS].to_numpy(dtype=float,copy=True)
        missing=~np.isfinite(previous).all(1)
        previous[missing]=current_volume[missing]
        for index,reservoir in enumerate(RESERVOIRS):
            delta=(current_volume[:,index]-previous[:,index])/CAPACITY[reservoir]
            result[f"history_volume_fraction_change_{days}d__{reservoir}"]=np.repeat(delta[:,None],HOURS,axis=1)
    current_value=tables["water_values"].reindex(starts)[RESERVOIRS].to_numpy(dtype=float)
    for index,reservoir in enumerate(RESERVOIRS):
        result[f"history_water_value_horizon_change__{reservoir}"]=base[f"terminal_value__{reservoir}"].to_numpy().reshape(-1,HOURS)-current_value[:,index,None]
    prior_volume=tables["volumes"].reindex(starts-pd.Timedelta(days=1))[RESERVOIRS].to_numpy(dtype=float)
    prior_times=pd.DatetimeIndex(np.repeat(starts.to_numpy(),24))+pd.to_timedelta(np.tile(np.arange(-24,0),len(starts)),unit="h")
    prior_flow=tables["inflows"].reindex(prior_times)[RESERVOIRS].to_numpy(dtype=float).reshape(len(starts),24,len(RESERVOIRS))
    valid=np.isfinite(prior_volume).all(1)&np.isfinite(prior_flow).all(axis=(1,2))
    # This residual also contains transfers, spill and storage rounding. It is
    # an available past water-balance descriptor, not observed generator discharge.
    residual=np.zeros_like(current_volume)
    residual[valid]=prior_flow[valid].sum(1)*3600/1e6-(current_volume[valid]-prior_volume[valid])
    for index,reservoir in enumerate(RESERVOIRS):
        result[f"history_net_non_natural_outflow_fraction__{reservoir}"]=np.repeat((residual[:,index]/CAPACITY[reservoir])[:,None],HOURS,axis=1)
    for group,reservoirs in [("intake",feeders),("upstream",UPSTREAM[generator])]:
        indices=[RESERVOIRS.index(r) for r in reservoirs]
        result[f"history_{group}_net_balance_flow_m3s"]=np.repeat((residual[:,indices].sum(1)*1e6/86400)[:,None],HOURS,axis=1)
    return result

def generator_features(base,generator):
    frame=reference_generator_features(base,generator)
    extra={}
    if "topology" in ADDITIONAL_GROUPS:extra.update(topology_features(base,generator))
    if "shape" in ADDITIONAL_GROUPS:extra.update(shape_features(base))
    if "physics" in ADDITIONAL_GROUPS:extra.update(physics_features(base,generator))
    if "history" in ADDITIONAL_GROUPS:extra.update(history_features(base,generator))
    if "allocation" in ADDITIONAL_GROUPS:extra.update(allocation_features(base,generator))
    if extra:
        frame=pd.concat([frame,pd.DataFrame({k:np.asarray(v,dtype=np.float32).reshape(-1) for k,v in extra.items()},index=frame.index)],axis=1)
    if not np.isfinite(frame.to_numpy()).all():raise ValueError("Nonfinite engineered features.")
    return frame.astype(np.float32,copy=False)
"""Development source inlined into the notebook: coarse deterministic hydraulic allocation features."""

ALLOCATION_STEP_HOURS = 12
ALLOCATION_CACHE = {}


def allocation_structure():
    """Build a sparse conservative-flow relaxation of the supplied directed graph."""
    from scipy.sparse import lil_matrix
    plants = list(PLANT_GENERATORS)
    nodes = sorted(set(FORWARD) | {node for values in FORWARD.values() for node in values})
    arcs = sorted((left, right) for left, values in FORWARD.items() for right in values)
    # Terminal river reaches and Hogga discharge into the external sink.
    arcs += [(node, ("sink", "outside")) for node in nodes if not FORWARD.get(node)]
    count = HOURS // ALLOCATION_STEP_HOURS
    nf, nr, nn = len(arcs), len(RESERVOIRS), len(nodes)
    node_position = {node: index for index, node in enumerate(nodes)}
    nv = count * (nf + 3 * nr)
    equality = lil_matrix((count * nn, nv), dtype=np.float64)
    inequality = lil_matrix((2 * count * nr, nv), dtype=np.float64)
    for step in range(count):
        for edge, (left, right) in enumerate(arcs):
            equality[step * nn + node_position[left], step * nf + edge] += 1
            if right in node_position:
                equality[step * nn + node_position[right], step * nf + edge] -= 1
        for r, reservoir in enumerate(RESERVOIRS):
            column = count * nf + step * nr + r
            row = step * nn + node_position[("reservoir", reservoir)]
            equality[row, column] += 1
            if step:
                equality[row, column - nr] -= 1
            # Soft upper/lower storage constraints keep exceptional raw cases
            # feasible; their violations are exposed as features, not concealed.
            upper, lower = 2 * (step * nr + r), 2 * (step * nr + r) + 1
            inequality[upper, column] = 1
            inequality[upper, count * (nf + nr) + step * nr + r] = -1
            inequality[lower, column] = -1
            inequality[lower, count * (nf + 2 * nr) + step * nr + r] = -1
    return {"plants": plants, "nodes": nodes, "arcs": arcs, "steps": count,
            "nf": nf, "nr": nr, "nn": nn, "nv": nv,
            "node_position": node_position, "eq": equality.tocsr(), "ub": inequality.tocsr()}


def allocation_case(start):
    """Use only S state, supplied planning inputs and S+168h values; never UC labels."""
    from scipy.optimize import linprog
    import warnings
    start = pd.Timestamp(start)
    if start in ALLOCATION_CACHE:
        return ALLOCATION_CACHE[start]
    if DEADLINE_UTC and datetime.now(timezone.utc) >= datetime.fromisoformat(DEADLINE_UTC):
        raise TimeoutError("Authorized preprocessing deadline reached.")
    global ALLOCATION_STRUCTURE
    if "ALLOCATION_STRUCTURE" not in globals():
        ALLOCATION_STRUCTURE = allocation_structure()
    s = ALLOCATION_STRUCTURE
    k, nf, nr, nn = s["steps"], s["nf"], s["nr"], s["nn"]
    times = pd.date_range(start, periods=HOURS, freq="h")
    volume = tables["volumes"].loc[start, RESERVOIRS].to_numpy(dtype=float)
    terminal = tables["water_values"].loc[start + pd.Timedelta(hours=HOURS), RESERVOIRS].to_numpy(dtype=float)
    price = tables["prices"].reindex(times).iloc[:, 0].to_numpy(dtype=float).reshape(k, -1).mean(1)
    inflow = tables["inflows"].reindex(times)
    minvol = tables["min_volume"].reindex(times)
    minflow = tables["min_flow"].reindex(times)
    seconds = ALLOCATION_STEP_HOURS * 3600
    qmax, factor = {}, {}
    levels = {r: np.interp(volume[i], *LEVEL_CURVES[r]) for i, r in enumerate(RESERVOIRS)}
    for plant in s["plants"]:
        feeders = PLANT_FEEDERS[plant]
        level = np.average([levels[r] for r in feeders], weights=[CAPACITY[r] for r in feeders])
        outlet = float(topology["model"]["plant"][plant]["outlet_line"])
        tail = max([outlet] + [levels[r] for r in RECEIVERS[plant]])
        head = np.array([max(level - tail, 1.0)])
        quantities = [float(efficiency(GEN_SPECS[g], head, float(GEN_SPECS[g]["p_max"]))[0][0])
                      for g in PLANT_GENERATORS[plant]]
        qmax[plant] = sum(quantities)
        factor[plant] = sum(float(GEN_SPECS[g]["p_max"]) for g in PLANT_GENERATORS[plant]) * 1e6 / (qmax[plant] * 3600)
    # Approximate local-energy-to-global-water conversion: preserve the receiving
    # water's value and add the local energy value of the first generating stage.
    # Initial-head factors and alternative routes make this a proxy, not SHOP's
    # optimized marginal values. Multiplying one local value by all downstream
    # energy would incorrectly create value by transferring to higher-value water.
    local_values = dict(zip(RESERVOIRS, terminal))
    global_values = {}
    def global_value(reservoir, seen=None):
        if reservoir in global_values:
            return global_values[reservoir]
        seen = set() if seen is None else seen
        if reservoir in seen:
            raise ValueError("Cycle in global-value approximation.")
        pending, visited, first_plants = list(FORWARD.get(("reservoir", reservoir), [])), set(), set()
        while pending:
            node = pending.pop()
            if node in visited:
                continue
            visited.add(node)
            if node[0] == "plant":
                first_plants.add(node[1])
            else:
                pending.extend(FORWARD.get(node, []))
        candidates = []
        for plant in first_plants:
            receiving = max([0.0] + [global_value(r, seen | {reservoir}) for r in RECEIVERS[plant]])
            candidates.append(float(local_values[reservoir]) * factor[plant] + receiving)
        global_values[reservoir] = max([0.0] + candidates)
        return global_values[reservoir]
    terminal_money = [global_value(r) for r in RESERVOIRS]
    objective = np.zeros(s["nv"])
    rhs = np.zeros(k * nn)
    limits = np.zeros(2 * k * nr)
    bounds = []
    plant_arc = {}
    for step in range(k):
        for edge, (left, right) in enumerate(s["arcs"]):
            low, high = 0.0, 1000.0 * seconds / 1e6
            if left[0] == "plant":
                high = qmax[left[1]] * seconds / 1e6
                objective[step * nf + edge] = -price[step] * factor[left[1]]
                plant_arc[left[1]] = edge
            if left[0] == "river" and left[1] in minflow:
                low = float(minflow[left[1]].to_numpy().reshape(k, -1)[step].sum()) * 3600 / 1e6
            bounds.append((low, max(high, low)))
        for node, position in s["node_position"].items():
            if node[1] in inflow and node[0] in {"reservoir", "river"}:
                rhs[step * nn + position] = float(inflow[node[1]].to_numpy().reshape(k, -1)[step].sum()) * 3600 / 1e6
        for index, reservoir in enumerate(RESERVOIRS):
            if step == 0:
                rhs[s["node_position"][("reservoir", reservoir)]] += volume[index]
            limits[2 * (step * nr + index)] = CAPACITY[reservoir]
            floor = float(minvol[reservoir].to_numpy().reshape(k, -1)[step].max()) if reservoir in minvol else 0.0
            limits[2 * (step * nr + index) + 1] = -floor
    bounds += [(0, None)] * (3 * k * nr)
    for index in range(nr):
        objective[k * nf + (k - 1) * nr + index] = -terminal_money[index]
    objective[k * (nf + nr):] = 1e6
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Unrecognized options detected")
        solved = linprog(objective, A_ub=s["ub"], b_ub=limits, A_eq=s["eq"], b_eq=rhs,
                         bounds=bounds, method="highs", options={"threads": 1})
    if solved.success:
        flow = solved.x[:k * nf].reshape(k, nf)
        storage = solved.x[k * nf:k * (nf + nr)].reshape(k, nr)
        production_fraction = np.column_stack([flow[:, plant_arc[p]] / (qmax[p] * seconds / 1e6) for p in s["plants"]])
        violations = solved.x[k * (nf + nr):].reshape(2, k, nr).sum(axis=(0, 2))
        equality_error = float(np.max(np.abs(s["eq"] @ solved.x - rhs)))
        if equality_error > 1e-5:
            raise ValueError("Allocation water-balance residual is too large.")
    else:
        production_fraction, storage, violations = np.zeros((k, len(s["plants"]))), np.tile(volume, (k, 1)), np.zeros(k)
        equality_error = None
    result = {"production_fraction": production_fraction, "storage": storage,
              "violations": violations, "success": bool(solved.success), "equality_error": equality_error}
    ALLOCATION_CACHE[start] = result
    return result


def allocation_features(base, generator):
    starts = base.attrs["case_starts"]
    results = [allocation_case(start) for start in starts]
    structure = ALLOCATION_STRUCTURE
    plant = PLANT_FOR[generator]
    position = structure["plants"].index(plant)
    flows = np.stack([r["production_fraction"] for r in results])
    storage = np.stack([r["storage"] for r in results])
    repeat = lambda value: np.repeat(value, ALLOCATION_STEP_HOURS, axis=1)
    result = {f"allocation_fraction__{p}": repeat(flows[:, :, i]) for i, p in enumerate(structure["plants"])}
    local = repeat(flows[:, :, position])
    result["allocation_local_fraction"] = local
    result["allocation_local_mean_fraction"] = np.repeat(local.mean(1)[:, None], HOURS, axis=1)
    result["allocation_local_remaining_mean_fraction"] = local[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1)
    result["allocation_local_elapsed_mean_fraction"] = local.cumsum(1) / np.arange(1, HOURS + 1)
    result["allocation_local_hours_positive_remaining"] = (local[:, ::-1] > 1e-7).cumsum(1)[:, ::-1]
    for label, reservoirs in [("intake", FEEDERS[generator]), ("upstream", UPSTREAM[generator]),
                              ("receiver", RECEIVERS[plant])]:
        if reservoirs:
            indices = [RESERVOIRS.index(r) for r in reservoirs]
            fraction = storage[:, :, indices].sum(2) / sum(CAPACITY[r] for r in reservoirs)
        else:
            fraction = np.zeros(storage.shape[:2])
        result[f"allocation_projected_{label}_fraction"] = repeat(fraction)
    result["allocation_soft_constraint_violation_Mm3"] = repeat(np.stack([r["violations"] for r in results]))
    result["allocation_failed"] = np.repeat(np.array([not r["success"] for r in results], dtype=float)[:, None], HOURS, axis=1)
    return result

In [ ]:
def chronological_split(start, end):
    start, end = pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC")
    if start >= end or end > HOLDOUT_START:
        raise ValueError("Development folds must end by the reserved holdout start.")
    horizon_end = train_cases["starttime"] + pd.Timedelta(hours=HOURS)
    fitting = train_cases.loc[horizon_end <= start].copy()
    validation = train_cases.loc[(train_cases["starttime"] >= start) & (horizon_end <= end)].copy()
    if fitting.empty or validation.empty:
        raise ValueError("Empty chronological fold.")
    if (fitting["starttime"] + pd.Timedelta(hours=HOURS)).max() > validation["starttime"].min():
        raise ValueError("Training and validation horizons overlap.")
    return fitting, validation

def even_sample(cases, maximum):
    indices = np.linspace(0, len(cases) - 1, min(maximum, len(cases)), dtype=int)
    return cases.iloc[indices].copy()

def assemble_predictions(cases, predictions):
    if set(predictions) != set(GENERATORS):
        raise ValueError("Predictions must cover all 14 generators.")
    values = np.empty((len(cases), len(LABELS)), dtype=np.float32)
    for generator in GENERATORS:
        probability = np.asarray(predictions[generator])
        if probability.shape != (len(cases), HOURS):
            raise ValueError(f"Wrong prediction shape for {generator}")
        values[:, TARGET_POSITIONS[generator]] = probability
    if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
        raise ValueError("Predictions must be finite probabilities in [0, 1].")
    result = pd.DataFrame(values, columns=LABELS)
    result.insert(0, "Run No", cases["Run No"].to_numpy())
    return result

display(pd.DataFrame([{"validation_start": start, "validation_end_exclusive": end,
                       "training_cases": len(chronological_split(start, end)[0]),
                       "validation_cases": len(chronological_split(start, end)[1])}
                      for start, end in FOLDS]))
probe_cases = pd.concat([train_cases.iloc[[0, 1]], test_cases.iloc[[0, -1]]], ignore_index=True)
probe = feature_matrix(probe_cases)
probe_single = feature_matrix(probe_cases.iloc[[1]])
np.testing.assert_allclose(probe.iloc[HOURS:2 * HOURS].to_numpy(), probe_single.to_numpy())
for i, start in enumerate(probe_cases["starttime"]):
    rows = slice(i * HOURS, (i + 1) * HOURS)
    times = pd.date_range(start, periods=HOURS, freq="h")
    np.testing.assert_allclose(probe.iloc[rows]["price"], lookup("prices", times).reshape(-1))
    for reservoir in RESERVOIRS:
        np.testing.assert_allclose(probe.iloc[rows][f"initial_volume__{reservoir}"], tables["volumes"].at[start, reservoir])
        np.testing.assert_allclose(probe.iloc[rows][f"terminal_value__{reservoir}"], tables["water_values"].at[start + pd.Timedelta(hours=HOURS), reservoir])
for g in GENERATORS:
    generator_features(probe, g)
    truth = labels_for(train_cases.iloc[:2], g).reshape(2, HOURS)
    np.testing.assert_array_equal(truth[:, [0, 24, 167]], train_cases.iloc[:2][[TARGET_COLUMNS[g][h] for h in (0, 24, 167)]])
roundtrip = assemble_predictions(train_cases.iloc[:2],
    {g: labels_for(train_cases.iloc[:2], g).reshape(2, HOURS) for g in GENERATORS})
np.testing.assert_array_equal(roundtrip[LABELS], train_cases.iloc[:2][LABELS])
NUMERIC_FEATURE_NAMES = generator_features(probe, GENERATORS[0]).columns.tolist()
FEATURE_NAMES = NUMERIC_FEATURE_NAMES + (["generator_id"] if MODEL_STRATEGY == "shared" else [])
MODEL_KEYS = ["shared"] if MODEL_STRATEGY == "shared" else GENERATORS
FEATURE_SIGNATURE = hashlib.sha256(json.dumps(FEATURE_NAMES).encode()).hexdigest()
if any(c in FEATURE_NAMES for c in ("Run No", "starttime", "total_calculation_time")):
    raise ValueError("Metadata leaked into predictors.")
bytes_per_case = HOURS * len(NUMERIC_FEATURE_NAMES) * np.dtype("float32").itemsize
print(f"Data checks passed: {len(FEATURE_NAMES)} features per generator-hour.")
print(f"One full training feature matrix: approximately {len(train_cases) * bytes_per_case / 2**20:.0f} MiB.")
print("This is not peak memory: copies, XGBoost histogram matrices, and model state add overhead.")
del probe, probe_single, roundtrip
if any(re.search(r'[\[\]{}":,]', name) for name in FEATURE_NAMES):
    raise ValueError("A feature name contains unsupported XGBoost JSON characters.")
print("XGBoost check mode audits preprocessing without fitting or importing XGBoost.")

In [ ]:
def safe_auc(actual, predicted):
    from sklearn.metrics import roc_auc_score
    actual = np.asarray(actual).reshape(-1)
    if np.unique(actual).size < 2:
        return None
    return float(roc_auc_score(actual, np.asarray(predicted).reshape(-1)))

def evaluate(cases, predictions):
    from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
    frame = assemble_predictions(cases, predictions)
    actual = cases[LABELS].to_numpy(dtype=np.uint8)
    predicted = frame[LABELS].to_numpy()
    overall = float(roc_auc_score(actual, predicted, average="micro"))
    detail = pd.DataFrame([{"generator": g,
        "auc": safe_auc(labels_for(cases, g), predictions[g]),
        "logloss": float(log_loss(labels_for(cases, g), predictions[g].reshape(-1), labels=[0, 1])),
        "brier": float(brier_score_loss(labels_for(cases, g), predictions[g].reshape(-1))),
        "observed_on_fraction": float(labels_for(cases, g).mean()),
        "mean_prediction": float(predictions[g].mean())} for g in GENERATORS])
    return overall, detail, frame

def seasonal_predictions(fitting, validation):
    groups = fitting[LABELS].groupby(fitting["starttime"].dt.month).mean()
    if not set(validation["starttime"].dt.month).issubset(groups.index):
        raise ValueError("The seasonal reference lacks a validation month in training.")
    values = groups.loc[validation["starttime"].dt.month]
    return {g: values[TARGET_COLUMNS[g]].to_numpy(dtype=np.float32) for g in GENERATORS}

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")

def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def fit_and_predict(fitting, evaluation, parameters, iterations, folder, *, early_stop):
    import xgboost as xgb
    started = perf_counter()
    folder.mkdir(parents=True, exist_ok=False)
    X_fit_base, X_eval_base = feature_matrix(fitting), feature_matrix(evaluation)
    feature_seconds = perf_counter()-started
    predicted, records, importance, curves = {}, [], [], {}
    for generator in GENERATORS:
        if DEADLINE_UTC and datetime.now(timezone.utc) >= datetime.fromisoformat(DEADLINE_UTC):
            raise TimeoutError("Authorized execution deadline reached.")
        generator_started = perf_counter()
        X_fit = generator_features(X_fit_base, generator)
        X_eval = generator_features(X_eval_base, generator)
        if X_fit.columns.tolist()!=FEATURE_NAMES or X_eval.columns.tolist()!=FEATURE_NAMES:
            raise ValueError("Partition feature schemas differ.")
        y=labels_for(fitting,generator)
        limit=int(iterations[generator] if isinstance(iterations,dict) else iterations)
        if limit<1:raise ValueError("Tree counts must be positive.")
        model=None
        if np.unique(y).size==1:
            constant=float(y[0])
            probability=np.full(len(evaluation)*HOURS,constant,dtype=np.float32)
            tree_count=1
            write_json(folder/f"{generator}_constant.json",{"probability":constant})
        else:
            constant=None
            options={"early_stopping_rounds":EARLY_STOPPING} if early_stop else {}
            model=xgb.XGBClassifier(**parameters,n_estimators=limit,random_state=SEED,n_jobs=THREADS,importance_type="total_gain",**options)
            print(f"Training {generator}: {len(y):,} rows, at most {limit} trees",flush=True)
            fit_options={"eval_set":[(X_eval,labels_for(evaluation,generator))],"verbose":100} if early_stop else {"verbose":False}
            model.fit(X_fit,y,**fit_options)
            tree_count=int(model.best_iteration+1) if early_stop else int(model.get_booster().num_boosted_rounds())
            if not 1<=tree_count<=limit:raise ValueError("Invalid retained tree count.")
            probability=model.predict_proba(X_eval,iteration_range=(0,tree_count),validate_features=True)[:,1].astype(np.float32)
            booster=model.get_booster()[:tree_count]
            booster.save_model(folder/f"{generator}.ubj")
            curves[generator]=model.evals_result() if early_stop else {}
            gains=booster.get_score(importance_type="total_gain")
            total=sum(gains.values())
            importance.extend({"generator":generator,"feature":name,"importance":100*float(gains.get(name,0))/total if total else 0,"gain":float(gains.get(name,0))} for name in FEATURE_NAMES)
        predicted[generator]=probability.reshape(len(evaluation),HOURS)
        records.append({"generator":generator,"trees":tree_count,"constant_probability":constant,"seconds":perf_counter()-generator_started})
        pd.DataFrame(records).to_csv(folder/"training_progress.csv",index=False)
        del X_fit,X_eval,model
        gc.collect()
    pd.DataFrame(importance).to_csv(folder/"feature_importance.csv",index=False)
    write_json(folder/"learning_curves.json",curves)
    write_json(folder/"timing.json",{"feature_seconds":feature_seconds,"total_seconds":perf_counter()-started})
    del X_fit_base,X_eval_base
    gc.collect()
    return predicted,records,importance

In [ ]:
RUN_DIRECTORY = None
run_summary = None
importance_rows = []
if RUN_MODE == "check":
    print("Check mode complete. No models trained and no submission written.")
else:
    if RUN_MODE in {"holdout", "submit"}:
        if not isinstance(SELECTED_CONFIG, dict):
            raise ValueError("Paste a completed validate run's configuration into SELECTED_CONFIG first.")
        if SELECTED_CONFIG.get("model_family") != MODEL_FAMILY:
            raise ValueError("Use a completed XGBoost configuration, not another-family configuration.")
        if SELECTED_CONFIG.get("feature_version") != FEATURE_VERSION or SELECTED_CONFIG.get("feature_signature") != FEATURE_SIGNATURE:
            raise ValueError("Frozen configuration and current features do not match.")
        if SELECTED_CONFIG.get("experiment", "baseline_logloss") != EXPERIMENT:
            raise ValueError("Set EXPERIMENT to the frozen configuration's experiment name before holdout or submit.")
        if SELECTED_CONFIG.get("seed") != SEED:
            raise ValueError("Use the same random seed as the frozen configuration.")
        if SELECTED_CONFIG.get("model_strategy", "per_generator") != MODEL_STRATEGY:
            raise ValueError("Frozen configuration uses a different model strategy.")
        if set(SELECTED_CONFIG.get("iterations", {})) != set(MODEL_KEYS):
            raise ValueError(f"Frozen configuration must contain tree counts for {MODEL_KEYS}.")
        if any(not isinstance(v, int) or v < 1 for v in SELECTED_CONFIG["iterations"].values()):
            raise ValueError("Frozen tree counts must be positive integers.")
        parameters = dict(SELECTED_CONFIG["parameters"])
        if parameters.get("device") != "cpu":
            raise ValueError("This notebook uses CPU XGBoost training.")
    else:
        parameters = dict(BASE_PARAMETERS)

    if set(parameters) & {"n_estimators", "num_iterations", "num_iteration", "num_boost_round",
                          "iterations", "random_state", "seed", "n_jobs", "num_threads",
                          "num_thread", "early_stopping_round", "early_stopping_rounds",
                          "early_stopping", "importance_type"}:
        raise ValueError("Keep tree budget, seed, threads and stopping outside the frozen parameter dictionary.")

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    RUN_DIRECTORY = ROOT / "outputs" / "xgboost" / f"{stamp}_{RUN_MODE}_{EXPERIMENT}"
    RUN_DIRECTORY.mkdir(parents=True, exist_ok=False)
    run_summary = {"status": "incomplete", "model_family": MODEL_FAMILY, "mode": RUN_MODE, "experiment": EXPERIMENT, "versions": VERSIONS,
        "seed": SEED, "threads": THREADS, "model_strategy": MODEL_STRATEGY, "feature_version": FEATURE_VERSION,
        "feature_signature": FEATURE_SIGNATURE, "features": FEATURE_NAMES,
        "parameters": parameters, "folds": FOLDS,
        "maximum_iterations": MAX_ITERATIONS, "early_stopping_rounds": EARLY_STOPPING,
        "selected_configuration": SELECTED_CONFIG,
        "input_sha256": {k: file_hash(v) for k, v in RAW_FILES.items()}}
    write_json(RUN_DIRECTORY / "run.json", run_summary)

    if RUN_MODE in {"smoke", "validate"}:
        fold_results, iteration_records = [], []
        active_folds = FOLDS[-1:] if RUN_MODE == "smoke" else FOLDS
        for start, end in active_folds:
            fitting, validation = chronological_split(start, end)
            if RUN_MODE == "smoke":
                fitting, validation = even_sample(fitting, 96), even_sample(validation, 48)
            folder = RUN_DIRECTORY / f"fold_{start[:4]}"
            predicted, records, importance = fit_and_predict(
                fitting, validation, parameters, 40 if RUN_MODE == "smoke" else MAX_ITERATIONS,
                folder, early_stop=True)
            auc, detail, prediction_frame = evaluate(validation, predicted)
            baseline_auc, _, _ = evaluate(validation, seasonal_predictions(fitting, validation))
            detail.to_csv(folder / "per_generator_metrics.csv", index=False)
            prediction_frame.to_csv(folder / "validation_predictions.csv", index=False)
            iteration_records.extend(records)
            importance_rows.extend(dict(row, fold=start[:4]) for row in importance)
            fold_results.append({"validation_start": start, "validation_end_exclusive": end,
                                 "training_cases": len(fitting), "validation_cases": len(validation),
                                 "micro_auc": auc, "seasonal_micro_auc": baseline_auc,
                                 "mean_generator_logloss": float(detail["logloss"].mean()),
                                 "mean_generator_brier": float(detail["brier"].mean())})
            print(f"{start[:4]} micro ROC-AUC: {auc:.6f}; same-fold seasonal reference: {baseline_auc:.6f}")
            display(detail)
            del predicted, prediction_frame
        run_summary["development_results"] = fold_results
        display(pd.DataFrame(fold_results))
        if RUN_MODE == "validate":
            selected = {"model_family": MODEL_FAMILY, "experiment": EXPERIMENT, "model_strategy": MODEL_STRATEGY, "feature_version": FEATURE_VERSION, "feature_signature": FEATURE_SIGNATURE,
                        "seed": SEED, "parameters": parameters,
                        "iterations": {g: max(1, int(np.median([r["trees"] for r in iteration_records if r["generator"] == g]))) for g in MODEL_KEYS}}
            write_json(RUN_DIRECTORY / "selected_configuration.json", selected)
            print("SELECTED_CONFIG for frozen-budget refits:")
            print(repr(selected))
            if any(r["trees"] >= MAX_ITERATIONS for r in iteration_records):
                print("Some models reached MAX_ITERATIONS. Inspect their learning curves before freezing the budget.")
    elif RUN_MODE == "holdout":
        fitting = train_cases.loc[train_cases["starttime"] + pd.Timedelta(hours=HOURS) <= HOLDOUT_START]
        holdout = train_cases.loc[train_cases["starttime"] >= HOLDOUT_START]
        if fitting.empty or holdout.empty:
            raise ValueError("Empty holdout partition.")
        predicted, _, importance_rows = fit_and_predict(fitting, holdout, parameters,
            SELECTED_CONFIG["iterations"], RUN_DIRECTORY / "holdout", early_stop=False)
        auc, detail, prediction_frame = evaluate(holdout, predicted)
        prediction_frame.to_csv(RUN_DIRECTORY / "holdout_predictions.csv", index=False)
        detail.to_csv(RUN_DIRECTORY / "holdout_metrics.csv", index=False)
        run_summary["holdout_micro_auc"] = auc
        print(f"Frozen-configuration holdout micro ROC-AUC: {auc:.6f}")
        display(detail)
    elif RUN_MODE == "submit":
        predicted, _, importance_rows = fit_and_predict(train_cases, test_cases, parameters,
            SELECTED_CONFIG["iterations"], RUN_DIRECTORY / "full_training", early_stop=False)
        submission = assemble_predictions(test_cases, predicted)
        if not submission.columns.equals(template.columns) or not submission["Run No"].equals(template["Run No"]):
            raise ValueError("Submission IDs or column order differ from the template.")
        if submission.shape != template.shape:
            raise ValueError("Submission shape differs from the template.")
        destination = RUN_DIRECTORY / "xgboost_submission.csv"
        submission.to_csv(destination, index=False)
        reloaded = pd.read_csv(destination)
        np.testing.assert_allclose(reloaded[LABELS], submission[LABELS], rtol=1e-6, atol=1e-8)
        np.testing.assert_array_equal(reloaded["Run No"], template["Run No"])
        run_summary["submission"] = str(destination.relative_to(ROOT))
        print(f"Verified submission saved: {destination}")

    run_summary["status"] = "complete"
    run_summary["total_seconds"] = perf_counter() - NOTEBOOK_STARTED
    write_json(RUN_DIRECTORY / "run.json", run_summary)
    (RUN_DIRECTORY / "environment.txt").write_text(
        "\n".join(f"{k}=={v}" for k, v in VERSIONS.items()) + "\n", encoding="utf-8")
    print(f"Recorded runtime so far: {run_summary['total_seconds'] / 60:.1f} minutes")
    print("Run artifacts:", RUN_DIRECTORY)

In [ ]:
if importance_rows:
    import matplotlib.pyplot as plt
    importance_table = pd.DataFrame(importance_rows)
    importance_table.to_csv(RUN_DIRECTORY / "feature_importance_all.csv", index=False)
    top = importance_table.groupby("feature")["importance"].mean().nlargest(20).sort_values()
    fig, ax = plt.subplots(figsize=(10, 7))
    top.plot.barh(ax=ax)
    ax.set(title="Mean feature importance across trained generator models",
           xlabel="Mean normalized split gain (%)", ylabel="Feature")
    fig.tight_layout()
    fig.savefig(RUN_DIRECTORY / "feature_importance.png", dpi=150)
    plt.show()
if RUN_DIRECTORY is not None:
    run_summary["total_seconds"] = perf_counter() - NOTEBOOK_STARTED
    write_json(RUN_DIRECTORY / "run.json", run_summary)
    print(f"Complete notebook runtime: {run_summary['total_seconds'] / 60:.1f} minutes")
else:
    print("No training results yet. Change RUN_MODE yourself when ready.")